# 8. Ускоренный поиск попадания в ФЭУ

Параметры детектора: `R=15 м`, `N=2000`, `r=0.1 м`, `a=0.5`. Фотоумножители распределены по сфере методом Фибоначчи. Для поиска кандидатов используется пространственная хеш-сетка: вместо проверки всех ФЭУ рассматриваются фотоумножители из соседних ячеек, после чего выполняется точная проверка касательного фотокатода. Последняя ячейка сравнивает алгоритм с полным перебором на одинаковых точках поверхности.

In [1]:
from math import cos, pi, sin, sqrt
import timeit
import numpy as np

class Detector:
    def __init__(self):
        self.R = 15.0
        self.N = 2000
        self.r = 0.1
        self.a = 0.5
        self.pmt_normals = self._make_pmt_normals()
        self.pmt_positions = self.R * self.pmt_normals

        max_chord_distance = sqrt(
            2 * self.R * (self.R - sqrt(self.R**2 - self.r**2))
        )
        self.cell_size = 2 * max_chord_distance
        self.spatial_index = {}
        for pmt_index, position in enumerate(self.pmt_positions):
            cell = self._cell_for(position)
            self.spatial_index.setdefault(cell, []).append(pmt_index)

    def _make_pmt_normals(self):
        indices = np.arange(self.N)
        z = 1.0 - 2.0 * (indices + 0.5) / self.N
        azimuth = pi * (1.0 + sqrt(5.0)) * indices
        radial = np.sqrt(1.0 - z**2)
        return np.column_stack((radial * np.cos(azimuth), radial * np.sin(azimuth), z))

    def _cell_for(self, point):
        return tuple(np.floor(np.asarray(point) / self.cell_size).astype(np.int64))

    def find_pmt_linear(self, surface_point):
        point = np.asarray(surface_point, dtype=float)
        if point.shape != (3,) or abs(np.linalg.norm(point) - self.R) > 1e-6:
            return None

        point_squared = np.dot(point, point)
        for pmt_index, normal in enumerate(self.pmt_normals):
            projection = np.dot(point, normal)
            if projection > 0 and point_squared - projection**2 <= self.r**2:
                return pmt_index
        return None

    def find_pmt_indexed(self, surface_point):
        point = np.asarray(surface_point, dtype=float)
        if point.shape != (3,) or abs(np.linalg.norm(point) - self.R) > 1e-6:
            return None

        base_cell = self._cell_for(point)
        candidates = []
        for dx in (-1, 0, 1):
            for dy in (-1, 0, 1):
                for dz in (-1, 0, 1):
                    cell = (base_cell[0] + dx, base_cell[1] + dy, base_cell[2] + dz)
                    candidates.extend(self.spatial_index.get(cell, ()))

        point_squared = np.dot(point, point)
        for pmt_index in candidates:
            normal = self.pmt_normals[pmt_index]
            projection = np.dot(point, normal)
            if projection > 0 and point_squared - projection**2 <= self.r**2:
                return pmt_index
        return None

detector = Detector()
print(f"ФЭУ: {detector.N}, радиус детектора: {detector.R:g} м, радиус катода: {detector.r:g} м")
print(f"Ячеек пространственного индекса: {len(detector.spatial_index)}")

ФЭУ: 2000, радиус детектора: 15 м, радиус катода: 0.1 м
Ячеек пространственного индекса: 2000


In [2]:
rng = np.random.default_rng(42)
random_directions = rng.normal(size=(1000, 3))
random_directions /= np.linalg.norm(random_directions, axis=1, keepdims=True)
random_surface_points = detector.R * random_directions
known_hit_points = detector.pmt_positions[:20]
test_points = np.vstack((random_surface_points, known_hit_points))

linear_results = [detector.find_pmt_linear(point) for point in test_points]
indexed_results = [detector.find_pmt_indexed(point) for point in test_points]
assert linear_results == indexed_results
print(f"Точек проверено: {len(test_points)}")
print(f"Найдено попаданий: {sum(index is not None for index in indexed_results)}")
print("Индексированный поиск совпадает с полным перебором на всех тестовых точках.")

def median_runtime(function, repeats=5):
    return float(np.median(timeit.repeat(function, repeat=repeats, number=1)))

linear_time = median_runtime(
    lambda: [detector.find_pmt_linear(point) for point in test_points]
)
indexed_time = median_runtime(
    lambda: [detector.find_pmt_indexed(point) for point in test_points]
)
speedup = linear_time / indexed_time
print(f"\nМедиана 5 запусков для {len(test_points)} точек:")
print(f"Полный перебор:       {linear_time * 1000:.2f} мс")
print(f"Пространственная сетка: {indexed_time * 1000:.2f} мс")
print(f"Ускорение: {speedup:.1f}x")

Точек проверено: 1020
Найдено попаданий: 47
Индексированный поиск совпадает с полным перебором на всех тестовых точках.

Медиана 5 запусков для 1020 точек:
Полный перебор:       2298.57 мс
Пространственная сетка: 14.67 мс
Ускорение: 156.7x
